# Cirrus Model Training on Google Colab

This notebook trains the Cirrus language model on C4 dataset.

In [ ]:
# Mount Google Drive (optional - for saving checkpoints)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Clone the Cirrus repository
%cd /content
!git clone https://github.com/YOUR_GITHUB_USERNAME/cirrus.git  # Or upload manually
OR upload the cirrus folder manually

In [ ]:
# Install dependencies
!pip install torch transformers datasets

In [ ]:
# Check GPU
import torch
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")
print(f"CUDA available: {torch.cuda.is_available()}")

In [ ]:
# Quick training test
import sys
sys.path.insert(0, '/content/cirrus')

import torch
import torch.nn as nn
from cirrus import CirrusModel, CirrusConfig
from transformers import AutoTokenizer
from datasets import load_dataset

# Model - use 'small' for GPU
config = CirrusConfig.small()
tokenizer = AutoTokenizer.from_pretrained("gpt2")
config.vocab_size = tokenizer.vocab_size
model = CirrusModel(config)
model = model.cuda()  # Move to GPU
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

print(f"Model params: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# Load C4 dataset
ds = load_dataset("allenai/c4", "en", split="train", streaming=True)
ds = ds.shuffle(buffer_size=10000)
print("Dataset loaded!")

In [ ]:
# Training loop
model.train()
step = 0
total_loss = 0

for batch in ds:
    text = batch.get("text", "")
    if len(text) < 50:
        continue
    
    tokens = tokenizer(text, return_tensors="pt", truncation=True, max_length=512)["input_ids"][0]
    if tokens.shape[0] < 10:
        continue
    
    # Move to GPU
    tokens = tokens.cuda()
    
    optimizer.zero_grad()
    logits, _, _, _ = model(tokens.unsqueeze(0))
    
    loss = nn.functional.cross_entropy(
        logits[:, :-1, :].reshape(-1, config.vocab_size),
        tokens[1:].reshape(-1),
        ignore_index=-1
    )
    
    loss.backward()
    optimizer.step()
    
    total_loss += loss.item()
    step += 1
    
    if step % 100 == 0:
        print(f"Step {step}, loss: {loss.item():.4f}, avg: {total_loss/step:.4f}")
    
    if step % 1000 == 0:
        torch.save(model.state_dict(), f'/content/cirrus_step{step}.pt')
        print(f"Saved checkpoint at step {step}")
    
    if step >= 10000:  # Stop after 10k steps
        break

In [ ]:
# Save final model
torch.save(model.state_dict(), '/content/cirrus_final.pt')
print("Training complete! Model saved to cirrus_final.pt")

In [ ]:
# Download the model
from google.colab import files
files.download('cirrus_final.pt')